# Qwen3-8B in 16 bit and in 4 bit

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

GB = 1024 ** 3

In [ ]:
model_id = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
question = "Explain the KV cache to a beginner in about 150 words."
messages = [{"role": "user", "content": question}]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    enable_thinking=False, return_tensors="pt"
).to("cuda")

In [ ]:
def answer(model):
    out = model.generate(**inputs, max_new_tokens=300, do_sample=False)
    new = out[0, inputs["input_ids"].shape[1]:]
    print(len(new), "new tokens")
    print(tokenizer.decode(new, skip_special_tokens=True))

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id, dtype=torch.bfloat16, device_map="cuda"
)
print(f"{torch.cuda.memory_allocated() / GB:.2f} GB")

In [ ]:
answer(model)

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()
print(f"{torch.cuda.memory_allocated() / GB:.2f} GB")

In [ ]:
config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=config_4bit,
    dtype=torch.bfloat16, device_map="cuda"
)
print(f"{torch.cuda.memory_allocated() / GB:.2f} GB")

In [ ]:
answer(model)

In [ ]:
q = model.model.layers[0].self_attn.q_proj.weight
print(q.dtype, q.shape)
print(model.lm_head.weight.dtype)